In [1]:
import torch, os, json, time, random
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
import matplotlib.pyplot as plt

from dataset_class import MultiTaskObjectDetectionDataset, collate_fn
from modelos_scratch import build_fasterrcnn_model

def compute_attr_acc(preds, targets):
    correct = (preds == targets).sum().item()
    return correct / len(targets) if len(targets) > 0 else 0

def compute_iou_torch(boxes1, boxes2):
    area1 = (boxes1[:, 2]-boxes1[:, 0])*(boxes1[:, 3]-boxes1[:, 1])
    area2 = (boxes2[:, 2]-boxes2[:, 0])*(boxes2[:, 3]-boxes2[:, 1])
    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0]*wh[:, :, 1]
    union = area1[:, None]+area2-inter
    return inter / (union + 1e-6)

def train_one_model(model, model_name, train_loader, val_loader, device, num_epochs=60, patience=5):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    history = []
    best_loss = float("inf")
    no_improve = 0

    for epoch in range(num_epochs):
        epoch_start = time.time()
        model.train()
        total_loss = 0
        batch_times = []
        total_batches = len(train_loader)

        for batch_idx, (images, targets, attrs) in enumerate(train_loader, start=1):
            batch_start = time.time()
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            attrs = [{k: v.to(device) for k, v in a.items()} for a in attrs]

            loss_dict = model(images, targets, attrs)
            loss = sum(loss_dict.values())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_time = time.time() - batch_start
            batch_times.append(batch_time)
            total_loss += loss.item()
            print(f"Epoch {epoch+1}/{num_epochs} - Batch {batch_idx}/{total_batches} - Batch Time: {batch_time:.2f}s")

        epoch_time = time.time() - epoch_start
        avg_loss = total_loss / len(train_loader)

        val_metrics = evaluate_model(model, val_loader, device)
        history.append({
            "epoch": epoch + 1,
            "train_loss": avg_loss,
            "val_loss": val_metrics["val_loss"],
            "acc_weather": val_metrics["acc_weather"],
            "acc_scene": val_metrics["acc_scene"],
            "acc_time": val_metrics["acc_time"],
            "detection_accuracy": val_metrics["detection_accuracy"],
            "avg_detections": val_metrics["avg_detections"],
            "epoch_time_seconds": epoch_time
        })

        print(f"[{model_name}] Epoch {epoch + 1}: "
              f"Train loss: {avg_loss:.4f}, Val loss: {val_metrics['val_loss']:.4f}, "
              f"W: {val_metrics['acc_weather']:.2%}, S: {val_metrics['acc_scene']:.2%}, "
              f"T: {val_metrics['acc_time']:.2%}, DetAcc: {val_metrics['detection_accuracy']:.2%}, "
              f"Epoch time: {epoch_time:.2f}s")

        if val_metrics["val_loss"] < best_loss:
            best_loss = val_metrics["val_loss"]
            no_improve = 0
            torch.save(model.state_dict(), f"best_{model_name}.pth")
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping!")
                break

    os.makedirs("logs", exist_ok=True)
    with open(f"logs/training_log_{model_name}.json", "w") as f:
        json.dump(history, f, indent=2)

    return history

def evaluate_model(model, val_loader, device):
    model.eval()
    val_loss = 0
    det_correct = det_total = det_count = 0
    weather_true, weather_pred = [], []
    scene_true, scene_pred = [], []
    time_true, time_pred = [], []

    with torch.no_grad():
        for images, targets, attrs in val_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            attrs = [{k: v.to(device) for k, v in a.items()} for a in attrs]

            loss_dict = model(images, targets, attrs)
            val_loss += sum(loss_dict.values()).item()

            detections, attr_logits = model(images)

            weather_pred += torch.argmax(attr_logits["weather"], dim=1).cpu().tolist()
            scene_pred += torch.argmax(attr_logits["scene"], dim=1).cpu().tolist()
            time_pred += torch.argmax(attr_logits["timeofday"], dim=1).cpu().tolist()

            weather_true += [a["weather"].item() for a in attrs]
            scene_true += [a["scene"].item() for a in attrs]
            time_true += [a["timeofday"].item() for a in attrs]

            for i, det in enumerate(detections):
                gt_boxes = targets[i]["boxes"]
                gt_labels = targets[i]["labels"]
                det_total += len(gt_boxes)

                pred_boxes = det["boxes"]
                pred_labels = det["labels"]
                pred_scores = det["scores"]
                keep = pred_scores >= 0.6
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0: continue

                ious = compute_iou_torch(gt_boxes, pred_boxes)
                max_iou, idx = ious.max(dim=1)
                matches = max_iou >= 0.5
                det_count += matches.sum().item()
                det_correct += (pred_labels[idx[matches]] == gt_labels[matches]).sum().item()

    return {
        "val_loss": val_loss / len(val_loader),
        "acc_weather": compute_attr_acc(torch.tensor(weather_pred), torch.tensor(weather_true)),
        "acc_scene": compute_attr_acc(torch.tensor(scene_pred), torch.tensor(scene_true)),
        "acc_time": compute_attr_acc(torch.tensor(time_pred), torch.tensor(time_true)),
        "detection_accuracy": det_correct / det_total if det_total else 0,
        "avg_detections": det_count / len(val_loader.dataset)
    }

# Dataset e treino
transform = transforms.Compose([transforms.ToTensor()])
datasets, loaders = {}, {}
for split in ["train", "val"]:
    ds = MultiTaskObjectDetectionDataset(f"images/{split}", f"labels/{split}", transform)
    indices = random.sample(range(len(ds)), max(1, int(len(ds)*0.5)))
    subset = Subset(ds, indices)
    loaders[split] = DataLoader(subset, batch_size=8, shuffle=(split=="train"), collate_fn=collate_fn)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbones = ["simplecnn", "midcnn", "deepcnn"]
results = {}

for backbone in backbones:
    print(f"Training Faster R-CNN with backbone: {backbone}")
    model = build_fasterrcnn_model(backbone)
    history = train_one_model(model, backbone, loaders["train"], loaders["val"], device)
    results[backbone] = history

# Plots
plt.figure(figsize=(10,5))
for backbone in backbones:
    plt.plot([x["val_loss"] for x in results[backbone]], label=f"{backbone} val loss")
plt.legend()
plt.grid()
plt.savefig("logs/fasterrcnn_comparison.png")
plt.show()


Todas as imagens possuem JSON correspondente.
Todas as imagens possuem JSON correspondente.
Training Faster R-CNN with backbone: simplecnn
Epoch 1/60 - Batch 1/438 - Batch Time: 6.36s
Epoch 1/60 - Batch 2/438 - Batch Time: 6.38s
Epoch 1/60 - Batch 3/438 - Batch Time: 6.38s
Epoch 1/60 - Batch 4/438 - Batch Time: 6.28s
Epoch 1/60 - Batch 5/438 - Batch Time: 5.71s
Epoch 1/60 - Batch 6/438 - Batch Time: 4.38s
Epoch 1/60 - Batch 7/438 - Batch Time: 4.22s
Epoch 1/60 - Batch 8/438 - Batch Time: 4.18s
Epoch 1/60 - Batch 9/438 - Batch Time: 4.01s
Epoch 1/60 - Batch 10/438 - Batch Time: 3.95s
Epoch 1/60 - Batch 11/438 - Batch Time: 3.83s
Epoch 1/60 - Batch 12/438 - Batch Time: 4.06s
Epoch 1/60 - Batch 13/438 - Batch Time: 4.08s
Epoch 1/60 - Batch 14/438 - Batch Time: 3.91s
Epoch 1/60 - Batch 15/438 - Batch Time: 3.79s
Epoch 1/60 - Batch 16/438 - Batch Time: 3.84s
Epoch 1/60 - Batch 17/438 - Batch Time: 3.78s
Epoch 1/60 - Batch 18/438 - Batch Time: 3.98s
Epoch 1/60 - Batch 19/438 - Batch Time: 4.

KeyboardInterrupt: 